# 00_2 Estimate LLM Screening Cost

Estimate the cost of running the notebook 01 LLM screen over all candidate `(PMID, pathogen, antibiotic, year)` rows.

This notebook uses the existing upstream parquet files, especially `candidate_amr_pair_year_counts.parquet` and `pmid_abstracts.parquet`, to sample prompts, estimate input tokens, and extrapolate total API cost. It does not call an LLM.

Pricing defaults are for `gpt-5-nano`, the model used in notebook 01. As of 2026-06-08, the official OpenAI pricing page lists `gpt-5-nano` at `$0.05 / 1M` input tokens, `$0.005 / 1M` cached input tokens, and `$0.40 / 1M` output tokens. Check https://platform.openai.com/docs/pricing before using the estimate for budgeting.


In [ ]:
from pathlib import Path
import sys

import pandas as pd

PROJECT_ROOT = Path.cwd().resolve()
if not (PROJECT_ROOT / "amr").exists():
    PROJECT_ROOT = PROJECT_ROOT.parent
if not (PROJECT_ROOT / "amr").exists():
    raise RuntimeError(f"Could not find repository root from {Path.cwd()}")
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from amr.screening import LLM_CLASSIFICATION_MAX_TOKENS, UNIFIED_AMR_INTEREST_SYSTEM_PROMPT, build_user_prompt, prepare_yearly_candidate_pmids


In [ ]:
OUTPUT_DIR = PROJECT_ROOT / "outputs"
PARQUET_OUTPUT_DIR = OUTPUT_DIR / "parquets"
DATA_DIR = PARQUET_OUTPUT_DIR

YEARLY_CANDIDATE_PATH = DATA_DIR / "candidate_amr_pair_year_counts.parquet"
ABSTRACT_CACHE_PATH = DATA_DIR / "pmid_abstracts.parquet"
UNIFIED_DECISIONS_PATH = DATA_DIR / "yearly_amr_unified_decisions.parquet"
ARCHIVED_DECISIONS_GLOB = OUTPUT_DIR / "archive" / "*" / "yearly_amr_unified_decisions.parquet"

MODEL_NAME = "gpt-5-nano"

# Pricing source checked on 2026-06-08:
# https://platform.openai.com/docs/pricing
# Model-specific details are also listed at:
# https://platform.openai.com/docs/models/gpt-5-nano
# Prices below are USD per 1M text tokens for gpt-5-nano at the time checked.
INPUT_USD_PER_1M_TOKENS = 0.05
OUTPUT_USD_PER_1M_TOKENS = 0.40
CACHED_INPUT_USD_PER_1M_TOKENS = 0.005

# If you use the OpenAI Batch API for this offline classification job, pricing is typically discounted.
# Keep this at 1.0 for ordinary synchronous API calls.
BATCH_API_MULTIPLIER = 1.0

# Token estimation controls.
SAMPLE_SIZE = 1000
RANDOM_SEED = 42

# Output tokens cannot be known before the model runs. If a compatible decisions parquet exists,
# the notebook estimates output size from compact reconstructed JSON rows. Otherwise this fallback is used.
ASSUMED_OUTPUT_TOKENS_PER_CALL = 140

# Set True only if you expect prompt-cache pricing to apply. This workflow has unique abstracts per request,
# so the conservative default is standard input pricing.
USE_CACHED_INPUT_PRICE = False


## Load Candidate Rows and Abstracts

The LLM is called at candidate-row granularity: one row per `(PMID, pathogen, antibiotic, year)`. Rows without abstracts receive deterministic defaults in notebook 01 and do not call the LLM, so they are excluded from the costed call count.


In [ ]:
if not YEARLY_CANDIDATE_PATH.exists():
    raise FileNotFoundError(f"Missing {YEARLY_CANDIDATE_PATH}. Run notebook 01 collection first.")
if not ABSTRACT_CACHE_PATH.exists():
    raise FileNotFoundError(f"Missing {ABSTRACT_CACHE_PATH}. Run notebook 01 abstract prefetch/filtering first.")

yearly_candidate_df = pd.read_parquet(YEARLY_CANDIDATE_PATH)
candidate_df = prepare_yearly_candidate_pmids(yearly_candidate_df)
abstract_df = pd.read_parquet(ABSTRACT_CACHE_PATH)
required_abstract_cols = {"pmid", "title", "abstract"}
missing_abstract_cols = required_abstract_cols - set(abstract_df.columns)
if missing_abstract_cols:
    raise KeyError(f"{ABSTRACT_CACHE_PATH} missing required columns: {sorted(missing_abstract_cols)}")

abstract_df = abstract_df[["pmid", "title", "abstract"]].copy()
abstract_df["pmid"] = abstract_df["pmid"].astype(str)
abstract_df["title"] = abstract_df["title"].fillna("").astype(str)
abstract_df["abstract"] = abstract_df["abstract"].fillna("").astype(str)

review_df = candidate_df.merge(abstract_df, on="pmid", how="left")
review_df["title"] = review_df["title"].fillna("").astype(str)
review_df["abstract"] = review_df["abstract"].fillna("").astype(str)
llm_call_df = review_df.loc[review_df["abstract"].str.strip().ne("")].copy()
no_abstract_df = review_df.loc[review_df["abstract"].str.strip().eq("")].copy()

summary_df = pd.DataFrame(
    [
        {"metric": "candidate_rows", "value": len(candidate_df)},
        {"metric": "rows_with_cached_abstract", "value": len(llm_call_df)},
        {"metric": "rows_without_abstract_or_cache", "value": len(no_abstract_df)},
        {"metric": "unique_pmids_in_candidates", "value": candidate_df["pmid"].nunique()},
        {"metric": "unique_pmids_with_cached_abstract", "value": llm_call_df["pmid"].nunique()},
    ]
)
summary_df


## Token Counter

`tiktoken` is used when available. If it is not installed, the notebook falls back to a conservative character-based estimate. Install `tiktoken` for better estimates.


In [ ]:
try:
    import tiktoken

    try:
        encoding = tiktoken.encoding_for_model(MODEL_NAME)
    except KeyError:
        encoding = tiktoken.get_encoding("o200k_base")

    TOKENIZER = f"tiktoken:{encoding.name}"

    def count_tokens(text: str) -> int:
        return len(encoding.encode(text or ""))

except Exception as exc:
    TOKENIZER = f"chars_per_4_fallback:{exc.__class__.__name__}"

    def count_tokens(text: str) -> int:
        return max(1, int(round(len(text or "") / 4)))

TOKENIZER


In [ ]:
def prompt_token_count(row: pd.Series) -> int:
    user_prompt = build_user_prompt(
        pmid=str(row["pmid"]),
        pathogen=str(row["pathogen"]),
        antibiotic=str(row["antibiotic"]),
        title=str(row.get("title", "")),
        abstract=str(row.get("abstract", "")),
    )
    # Chat message framing adds a small overhead. This keeps the estimate conservative without depending on API internals.
    message_overhead_tokens = 12
    return count_tokens(UNIFIED_AMR_INTEREST_SYSTEM_PROMPT) + count_tokens(user_prompt) + message_overhead_tokens

sample_n = min(SAMPLE_SIZE, len(llm_call_df))
if sample_n == 0:
    raise ValueError("No rows with abstracts were found; there are no LLM calls to cost from the current cache.")

sample_df = llm_call_df.sample(n=sample_n, random_state=RANDOM_SEED).copy()
sample_df["input_tokens"] = sample_df.apply(prompt_token_count, axis=1)

input_token_stats = sample_df["input_tokens"].describe(percentiles=[0.05, 0.25, 0.5, 0.75, 0.95]).to_frame("input_tokens")
input_token_stats


## Estimate Output Tokens

Before running the LLM, output tokens are only an estimate. This section tries to use an existing active or archived decisions parquet to infer the size of compact JSON outputs. If none is available, it uses `ASSUMED_OUTPUT_TOKENS_PER_CALL`.


In [ ]:
import json

def clean_bool(value: object, default: bool = False) -> bool:
    if pd.isna(value):
        return default
    if isinstance(value, bool):
        return value
    return str(value).strip().lower() in {"true", "1", "yes"}

def clean_list(value: object) -> list[str]:
    if isinstance(value, list):
        return [str(item) for item in value if str(item).strip()]
    if hasattr(value, "tolist"):
        return clean_list(value.tolist())
    return []

def clean_text(value: object, default: str = "") -> str:
    if pd.isna(value):
        return default
    return str(value)

def compact_json_for_decision(row: pd.Series) -> str:
    payload = {
        "decision": clean_text(row.get("unified_decision", row.get("decision", "uncertain")), "uncertain"),
        "primary_category": clean_text(row.get("amr_primary_category", "NONE"), "NONE"),
        "rationale": clean_text(row.get("unified_rationale", row.get("rationale", ""))),
        "evidence_phrases": clean_list(row.get("amr_evidence_phrases", [])),
        "target_pair_relation_supported": clean_bool(row.get("amr_target_pair_relation_supported", False)),
        "is_amr_relevant": clean_bool(row.get("amr_is_relevant", False)),
    }
    return json.dumps(payload, ensure_ascii=False, separators=(",", ":"))

candidate_decision_paths = []
if UNIFIED_DECISIONS_PATH.exists():
    candidate_decision_paths.append(UNIFIED_DECISIONS_PATH)
candidate_decision_paths.extend(sorted(OUTPUT_DIR.glob("archive/*/yearly_amr_unified_decisions.parquet")))

output_token_source = "fallback_assumption"
estimated_output_tokens_per_call = ASSUMED_OUTPUT_TOKENS_PER_CALL
output_token_stats = pd.DataFrame()

for decision_path in candidate_decision_paths:
    try:
        decisions_df = pd.read_parquet(decision_path)
    except Exception:
        continue
    if decisions_df.empty:
        continue
    sample_decisions = decisions_df.sample(n=min(SAMPLE_SIZE, len(decisions_df)), random_state=RANDOM_SEED).copy()
    token_counts = sample_decisions.apply(lambda row: count_tokens(compact_json_for_decision(row)), axis=1)
    if len(token_counts):
        estimated_output_tokens_per_call = float(token_counts.median())
        output_token_source = str(decision_path)
        output_token_stats = token_counts.describe(percentiles=[0.05, 0.25, 0.5, 0.75, 0.95]).to_frame("output_tokens")
        break

print(f"output_token_source: {output_token_source}")
print(f"estimated_output_tokens_per_call: {estimated_output_tokens_per_call:.1f}")
output_token_stats


## Extrapolate Total Cost

The sampled mean input tokens are multiplied by the number of rows that will actually call the LLM. Output tokens use the inferred or assumed per-call estimate above.


In [ ]:
input_price = CACHED_INPUT_USD_PER_1M_TOKENS if USE_CACHED_INPUT_PRICE else INPUT_USD_PER_1M_TOKENS
calls = len(llm_call_df)
mean_input_tokens = float(sample_df["input_tokens"].mean())
median_input_tokens = float(sample_df["input_tokens"].median())
p95_input_tokens = float(sample_df["input_tokens"].quantile(0.95))

total_input_tokens_mean = calls * mean_input_tokens
total_input_tokens_median = calls * median_input_tokens
total_input_tokens_p95 = calls * p95_input_tokens
total_output_tokens = calls * estimated_output_tokens_per_call

def usd(input_tokens: float, output_tokens: float) -> float:
    return BATCH_API_MULTIPLIER * (
        (input_tokens / 1_000_000) * input_price
        + (output_tokens / 1_000_000) * OUTPUT_USD_PER_1M_TOKENS
    )

cost_summary_df = pd.DataFrame(
    [
        {
            "scenario": "mean_input_tokens",
            "llm_calls": calls,
            "input_tokens_per_call": mean_input_tokens,
            "output_tokens_per_call": estimated_output_tokens_per_call,
            "total_input_tokens": total_input_tokens_mean,
            "total_output_tokens": total_output_tokens,
            "estimated_cost_usd": usd(total_input_tokens_mean, total_output_tokens),
        },
        {
            "scenario": "median_input_tokens",
            "llm_calls": calls,
            "input_tokens_per_call": median_input_tokens,
            "output_tokens_per_call": estimated_output_tokens_per_call,
            "total_input_tokens": total_input_tokens_median,
            "total_output_tokens": total_output_tokens,
            "estimated_cost_usd": usd(total_input_tokens_median, total_output_tokens),
        },
        {
            "scenario": "p95_input_tokens_upper_bound",
            "llm_calls": calls,
            "input_tokens_per_call": p95_input_tokens,
            "output_tokens_per_call": estimated_output_tokens_per_call,
            "total_input_tokens": total_input_tokens_p95,
            "total_output_tokens": total_output_tokens,
            "estimated_cost_usd": usd(total_input_tokens_p95, total_output_tokens),
        },
    ]
)

cost_summary_df["estimated_cost_usd"] = cost_summary_df["estimated_cost_usd"].round(4)
cost_summary_df


In [ ]:
print(f"Model: {MODEL_NAME}")
print(f"Tokenizer/estimator: {TOKENIZER}")
print(f"Input price used: ${input_price:.6f} per 1M tokens")
print(f"Output price used: ${OUTPUT_USD_PER_1M_TOKENS:.6f} per 1M tokens")
print(f"Batch multiplier: {BATCH_API_MULTIPLIER}")
print(f"Rows that call the LLM: {calls:,}")
print(f"Rows skipped because abstract is missing: {len(no_abstract_df):,}")
print(f"Mean-input estimated total: ${cost_summary_df.loc[cost_summary_df['scenario'].eq('mean_input_tokens'), 'estimated_cost_usd'].iloc[0]:,.4f}")


## Optional: Inspect Largest Estimated Prompts

Use this to sanity-check whether unusually long abstracts dominate the estimate.


In [ ]:
inspect_cols = ["pmid", "pathogen", "antibiotic", "year", "input_tokens", "title"]
sample_df.sort_values("input_tokens", ascending=False)[inspect_cols].head(20)
